In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve

from imblearn.over_sampling import SMOTE

# 读取数据
raw = pd.read_csv('../datasets/feature_engineered_pima.csv')
X = raw.drop(columns=['Outcome'])
y = raw['Outcome']

# 处理异常值：0 值替换为中位数(适用于 Glucose / BloodPressure / SkinThickness / Insulin / BMI)
zero_features = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in zero_features:
    X[col] = X[col].replace(0, np.nan)
    X[col].fillna(X[col].median(), inplace=True)

# 划分训练/测试
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# 过采样
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

# 模型与参数网格
pipelines = {
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(random_state=42, max_iter=1000, solver='liblinear'))]),
    'Random Forest': Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier(n_jobs=-1, random_state=42))]),
    'Gradient Boosting': Pipeline([('scaler', StandardScaler()), ('clf', GradientBoostingClassifier(random_state=42))])
}

param_grid = {
    'Logistic Regression': {
        'clf__C': [0.01, 0.1, 1, 10],
        'clf__penalty': ['l1', 'l2']
    },
    'Random Forest': {
        'clf__n_estimators': [100, 200, 500],
        'clf__max_depth': [None, 6, 10],
        'clf__min_samples_split': [2, 5]
    },
    'Gradient Boosting': {
        'clf__n_estimators': [100, 200],
        'clf__learning_rate': [0.01, 0.05, 0.1],
        'clf__max_depth': [3, 5]
    }
}

best_models = {}
results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, pipe in pipelines.items():
    grid = GridSearchCV(pipe, param_grid[name], cv=cv, scoring='f1', n_jobs=-1, verbose=1)
    grid.fit(X_train, y_train)

    best = grid.best_estimator_
    best_models[name] = best

    y_pred = best.predict(X_test)
    y_proba = best.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_proba)

    results.append({
        'Model': name,
        'BestParams': grid.best_params_,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC AUC': roc
    })

    print('\n-----', name, 'best model -----')
    print('best params:', grid.best_params_)
    print('accuracy:', acc)
    print('precision:', prec)
    print('recall:', rec)
    print('f1:', f1)
    print('roc_auc:', roc)
    print(classification_report(y_test, y_pred, digits=4))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'{name} Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.figure(figsize=(6, 4))
    plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc:.4f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{name} ROC Curve')
    plt.legend(loc='lower right')
    plt.show()

# 保存结果
results_df = pd.DataFrame(results).sort_values('F1-Score', ascending=False)
print('\n综合排序结果:')
print(results_df)

# 保存最佳模型
best_name = results_df.iloc[0]['Model']
joblib.dump(best_models[best_name], '../datasets/best_diabetes_model.pkl')
print('最佳模型已保存为', best_name)

# 特征重要性（RandomForest）
if 'Random Forest' in best_models:
    rf = best_models['Random Forest'].named_steps['clf']
    if hasattr(rf, 'feature_importances_'):
        sf = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
        print('\nRandomForest 特征重要性:')
        print(sf)

        plt.figure(figsize=(10, 6))
        sns.barplot(x=sf.values, y=sf.index, palette='viridis')
        plt.title('Random Forest Feature Importance')
        plt.xlabel('Importance')
        plt.ylabel('Feature')
        plt.show()


def predict_diabetes(input_dict):
    model = joblib.load('../datasets/best_diabetes_model.pkl')
    df = pd.DataFrame([input_dict])
    return int(model.predict(df)[0]), float(model.predict_proba(df)[0, 1])

# 示例
sample_input = {
    'Pregnancies': 2,
    'Glucose': 120,
    'BloodPressure': 70,
    'SkinThickness': 20,
    'Insulin': 80,
    'BMI': 28.0,
    'DiabetesPedigreeFunction': 0.5,
    'Age': 32
}
res, prob = predict_diabetes(sample_input)
print('示例预测(样本):', res, '概率:', round(prob, 4))


<ipython-input-1-75a65b8d1ba8>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X[col].fillna(X[col].median(), inplace=True)
<ipython-input-1-75a65b8d1ba8>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'd

UnicodeEncodeError: 'ascii' codec can't encode characters in position 18-20: ordinal not in range(128)

: 